In [1]:
import torch 
torch.cuda.is_available()

True

In [4]:
import os

import sys
sys.path.insert(0, "/staging/leuven/stg_00002/lcb/gpartel/software/deepSCENIC")

import torch
import torch.nn.functional as F
import numpy as np
import scanpy as sc
import pandas as pd
import argparse
import json

from src.deepSCENIC import deepSCENIC

### Load model

In [5]:
# Output dir of deepSCENIC
RES_PATH = '/lustre1/project/stg_00002/lcb/gpartel/projects/deepSCENIC/results/DeepSEM_out_500bp_prior_3/'
parser = argparse.ArgumentParser()
opt = parser.parse_args('')
with open(RES_PATH + "/hyperparma.txt", 'r') as f:
            opt.__dict__.update(json.load(f))

In [6]:
# Compile model
model = deepSCENIC(opt)
dataloader, _, TFs_idx, r2g_dist_coo, seq_dataloader, _ = model.init_data()

dir exist
reading data...
data read!
AnnData object with n_obs × n_vars = 9409 × 14298
    var: 'n_cells'
    uns: 'log1p'
    layers: 'log_norm', 'raw'
AnnData object with n_obs × n_vars = 9409 × 316633
False


### Extract TF2r (TF to region: E1) and r2g (region to gene: E2) matrix

In [ ]:
from src.model import VAE
from torch import nn
from torch.autograd import Variable
from src.tf2rNet.model import TF2rNet
from tqdm import tqdm

model.opt.split_test = False
model.opt.train = False
vae_model_path= RES_PATH + '/model.pth'
tf2r_model_path = RES_PATH + '/model_tf2r.pth'

if opt.device=='cuda':
    Tensor = torch.cuda.FloatTensor
elif opt.device=='cpu':
    Tensor = torch.FloatTensor

adj_E1s = []
for i in range(10):
    with torch.no_grad():
        # Initialize TF2rNet model
        model_dict = torch.load(tf2r_model_path,  map_location=torch.device(opt.device))['model_state_dict']
        tf2rNet = TF2rNet(opt.TF2rNet_bottleneck_size, len(TFs_idx)).float().to(opt.device)
        tf2rNet.load_state_dict(model_dict) # Load weights
        tf2rNet.eval()
        tf2rNet.dense_block1.dropout_layer.train() # Enable dropouts to estimate uncertanty
        tf2rNet.dense_block2.dropout_layer.train() # Enable dropouts to estimate uncertanty
        
        # TF2rNet forward pass
        tf_pred_l = []
        for j, (seq_data_batch, seq_data_batch_idx) in tqdm(enumerate(seq_dataloader, 0), total=len(seq_dataloader)):
            X = Variable(seq_data_batch[0].type(Tensor))
            tf_pred,_ = tf2rNet(X)
            tf_pred_l.append(tf_pred)# * (torch.sigmoid(tf_pred_mask)>0.5).float()*1)
        adj_E1s.append(torch.cat(tf_pred_l).T)
adj_E1s = torch.stack(adj_E1s)
        
with torch.no_grad():
    vae = VAE(TFs_idx, r2g_dist_coo, 1, model.opt.n_hidden, dev=opt.device).float()
    vae.load_state_dict(torch.load(vae_model_path,  map_location=torch.device('cpu'))['model_state_dict'])

 51%|█████     | 162/317 [00:01<00:01, 95.97it/s]

Compute mean and entropy of tf2r predictions

In [10]:
from scipy.stats import entropy
adj_E1s_mean = adj_E1s.mean(dim=0)
adj_E1s_entropy = entropy(torch.exp(adj_E1s).detach().cpu().numpy(), axis=0)

In [11]:
E1 = pd.DataFrame(adj_E1s_mean.detach().cpu().numpy(), columns=dataloader['atac_region_name'], index=np.array(dataloader['rna_gene_name'])[TFs_idx])
E1_entropy = pd.DataFrame(adj_E1s_entropy, columns=dataloader['atac_region_name'], index=np.array(dataloader['rna_gene_name'])[TFs_idx])

In [12]:
with torch.no_grad():
    E2 = torch.sparse_coo_tensor(vae.r2g_dist.indices().to(opt.device), torch.relu(vae.adj_E2), vae.r2g_dist.shape).to_dense()
    E2 = pd.DataFrame(E2.cpu().numpy(), columns=dataloader['rna_gene_name'], index=dataloader['atac_region_name'])

In [13]:
# Save matrices
E1.to_pickle(RES_PATH + 'E1.pkl')
E1_entropy.to_pickle(RES_PATH + 'E1_entropy.pkl')
E2.to_pickle(RES_PATH + 'E2.pkl')

### Building eGRN 

Threshold TF2r scores based on Otsu thresholding on each TF absolute scores

In [9]:
E1 = pd.read_pickle(RES_PATH + 'E1.pkl')
E2 = pd.read_pickle(RES_PATH + 'E2.pkl')

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm
from skimage import filters

def tf_df(E1, E2, tf):
    TFs, regions, target_genes, tf2r_scores, r2g_scores, tf2g_scores, tf2r_entropys = [], [], [], [], [], [], []
    if E1.loc[tf,:].abs()[E1.loc[tf,:]>0].shape[0]>0:
        tf2r_th = filters.threshold_otsu(E1.loc[tf,:].abs()[E1.loc[tf,:]>0])
        regs = E1.loc[tf,:].abs()[E1.loc[tf,:] > tf2r_th].index
        for r in regs:
            targets = E2.loc[r, :].index
            for g in targets:            
                TFs.append(tf)
                regions.append(r)
                target_genes.append(g)
                tf2r_scores.append(E1.loc[tf,r])
                tf2g_scores.append(E1.loc[tf,r]*E2.loc[r,g])
                r2g_scores.append(E2.loc[r,g])
                tf2r_entropys.append(E1_entropy.loc[tf,r])

    return TFs, regions, target_genes, tf2r_scores, tf2g_scores, r2g_scores, tf2r_entropys

res = Parallel(n_jobs=32)(delayed(tf_df)(E1, E2, tf) for tf in tqdm(E1.index))


  5%|▍         | 64/1360 [00:16<00:46, 27.77it/s]

In [ ]:
TFs, regions, target_genes, tf2r_scores, tf2g_scores, r2g_scores, tf2r_entropys = [], [], [], [], [], [], []
for i in tqdm(range(len(res))):
    TFs = TFs + res[i][0]
    regions = regions + res[i][1]
    target_genes = target_genes + res[i][2]
    tf2r_scores = tf2r_scores + res[i][3]
    tf2g_scores = tf2g_scores + res[i][4]
    r2g_scores = r2g_scores + res[i][5]
    tf2r_entropys = tf2r_entropys + res[i][6]

In [ ]:
# Building deepSCENIC dataframe (equivalent to SCENIC+ 'eRegulon_metadata' dataframe)
df_deepSCENIC = pd.DataFrame({'TF':TFs, 'region':regions, 'gene':target_genes, 'tf2r_score':tf2r_scores, 'tf2g_score':tf2g_scores , 'r2g_score':r2g_scores, 'tf2r_entropy':tf2r_entropys})
df_deepSCENIC['Consensus_name'] = [
        f"{TF}_+" if tf2r_score >= 0 else f"{TF}_-" for TF, tf2r_score in df_deepSCENIC[['TF', 'tf2r_score']].to_numpy()]
df_deepSCENIC

Threshold dataframe (Otsu) based on region to gene scores

In [ ]:
import matplotlib.pyplot as plt

df_deepSCENIC.loc[:,'r2g_score'].hist(bins=100);
th = filters.threshold_otsu(df_deepSCENIC.loc[:,'r2g_score'])
plt.axvline(th, color='r')
plt.yscale('log')
plt.title("R2G scores")

In [ ]:
df_deepSCENIC = df_deepSCENIC[df_deepSCENIC.r2g_score>th]

In [ ]:
# Saving dataframe
df_deepSCENIC.to_pickle(RES_PATH + '/df_deepSCENIC.pkl')